### Task 1: Designing a Heuristic for a Warehouse Robot

In [ ]:
import math
import networkx as nx
import matplotlib.pyplot as plt

locations = {
    "Receiving_Area": (0, 0),
    "Storage_A": (2, 1),
    "Storage_B": (1, 4),
    "Sorting_Area": (4, 2),
    "Inspection_Area": (5, 5),
    "Packing_Station": (7, 6)
}

warehouse_graph = {
    "Receiving_Area": {"Storage_A": 2.2, "Storage_B": 4.1},
    "Storage_A": {"Sorting_Area": 2.2},
    "Storage_B": {"Inspection_Area": 5.0, "Sorting_Area": 6.0},
    "Sorting_Area": {"Inspection_Area": 3.2, "Packing_Station": 6.0},
    "Inspection_Area": {"Packing_Station": 3.2},
    "Packing_Station": {}
}

def heuristic(current, goal):
    x1, y1 = locations[current]
    x2, y2 = locations[goal]
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

goal = "Packing_Station"

print("Heuristic Values")
print("----------------")
for location in locations:
    print(f"{location}: {heuristic(location, goal):.2f}")

#### NetworkX visualization: Since Task 1 does not ask GBFS/A* to find a path, you are supposed to highlight the actual minimum-cost path merely for visualization:

In [ ]:
G = nx.DiGraph()
for node, neighbors in warehouse_graph.items():
    for neighbor, weight in neighbors.items():
        G.add_edge(node, neighbor, weight=weight)

solution_path = nx.shortest_path(G, source="Receiving_Area", target="Packing_Station", weight="weight")
solution_cost = nx.shortest_path_length(G, source="Receiving_Area", target="Packing_Station", weight="weight")

print("\nMinimum cost path:")
print(" -> ".join(solution_path))
print(f"Total cost: {solution_cost:.2f}")

pos = locations
path_edges = list(zip(solution_path, solution_path[1:]))
node_colors = ["orange" if n in solution_path else "lightblue" for n in G.nodes()]
edge_colors = ["red" if e in path_edges else "gray" for e in G.edges()]

plt.figure(figsize=(12, 7))
nx.draw(G, pos, with_labels=True, node_color=node_colors, edge_color=edge_colors,
        node_size=2000, font_size=8, arrows=True)
edge_labels = nx.get_edge_attributes(G, "weight")
nx.draw_networkx_edge_labels(G, pos, edge_labels)
plt.title("Warehouse Robot: Weighted Graph")
plt.axis("off")
plt.show()

In [ ]:
print("\nHeuristic Condition Verification")
print("***********************************")
for node in warehouse_graph:
    for neighbor, cost in warehouse_graph[node].items():
        h_node = heuristic(node, goal)
        h_neighbor = heuristic(neighbor, goal)
        consistent = h_node <= cost + h_neighbor
        print(f"{node} -> {neighbor}: h(n)={h_node:.2f}, cost={cost}, h(n')={h_neighbor:.2f}, consistent={consistent}")

**Observation:** The Euclidean heuristic is consistent — for every edge, h(n) ≤ edge_cost + h(n'), confirming it never overestimates and satisfies the triangle inequality.

### Task 2: Greedy Best-First Search for Airport Baggage Handling

In [ ]:
import math
import heapq
import networkx as nx
import matplotlib.pyplot as plt

locations = {
    "Baggage_Area": (0, 0),
    "Security": (2, 1),
    "Checkpoint": (1, 4),
    "Food_Court": (4, 2),
    "Terminal_Hall": (5, 5),
    "Departure_Gate": (8, 6)
}

airport_graph = {
    "Baggage_Area": {"Security": 2.2, "Checkpoint": 4.1},
    "Security": {"Food_Court": 2.2},
    "Checkpoint": {"Terminal_Hall": 5.0},
    "Food_Court": {"Terminal_Hall": 3.2, "Departure_Gate": 6.0},
    "Terminal_Hall": {"Departure_Gate": 3.2},
    "Departure_Gate": {}
}

def heuristic(current, goal):
    x1, y1 = locations[current]
    x2, y2 = locations[goal]
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

def greedy_best_first_search(start, goal, graph):
    counter = 0
    open_set = []
    heapq.heappush(open_set, (heuristic(start, goal), counter, start))
    came_from = {start: None}
    visited = set()
    expansion_order = []

    while open_set:
        _, _, current = heapq.heappop(open_set)
        if current in visited:
            continue
        visited.add(current)
        expansion_order.append(current)
        if current == goal:
            break
        for neighbor in graph.get(current, {}):
            if neighbor not in visited:
                counter += 1
                heapq.heappush(open_set, (heuristic(neighbor, goal), counter, neighbor))
                if neighbor not in came_from:
                    came_from[neighbor] = current

    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = came_from.get(current)
    path.reverse()

    total_cost = sum(graph[path[i]][path[i+1]] for i in range(len(path)-1))
    return path, total_cost, expansion_order

start = "Baggage_Area"
goal = "Departure_Gate"
path, total_cost, expansion_order = greedy_best_first_search(start, goal, airport_graph)

print("GBFS")
print("--------------------------------")
print("Expansion order:")
print(" -> ".join(expansion_order))
print("\nSolution path:")
print(" -> ".join(path))
print(f"\nTotal path cost: {total_cost:.2f}")

#### NetworkX visualization

In [ ]:
G = nx.DiGraph()
for node, neighbors in airport_graph.items():
    for neighbor, weight in neighbors.items():
        G.add_edge(node, neighbor, weight=weight)

pos = locations
path_edges = list(zip(path, path[1:]))
node_colors = ["orange" if n in path else "lightblue" for n in G.nodes()]
edge_colors = ["red" if e in path_edges else "gray" for e in G.edges()]

plt.figure(figsize=(12, 7))
nx.draw(G, pos, with_labels=True, node_color=node_colors, edge_color=edge_colors,
        node_size=2000, font_size=8, arrows=True)
edge_labels = nx.get_edge_attributes(G, "weight")
nx.draw_networkx_edge_labels(G, pos, edge_labels)
plt.title("Airport Baggage Handling - GBFS")
plt.axis("off")
plt.show()

### Task 3: A* Search for Emergency Supply Robot

In [ ]:
import math
import heapq
import networkx as nx
import matplotlib.pyplot as plt

locations = {
    "Pharmacy": (0, 0),
    "Main_Corridor": (2, 1),
    "Patient_Wing": (1, 4),
    "Nursing_Station": (4, 2),
    "Laboratory": (5, 5),
    "Emergency_Ward": (8, 6)
}

hospital_graph = {
    "Pharmacy": {"Main_Corridor": 2.2, "Patient_Wing": 4.1},
    "Main_Corridor": {"Nursing_Station": 2.2},
    "Patient_Wing": {"Laboratory": 5.0},
    "Nursing_Station": {"Laboratory": 3.2, "Emergency_Ward": 6.0},
    "Laboratory": {"Emergency_Ward": 3.2},
    "Emergency_Ward": {}
}

def heuristic(current, goal):
    x1, y1 = locations[current]
    x2, y2 = locations[goal]
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

def reconstruct_path(came_from, current):
    path = []
    while current is not None:
        path.append(current)
        current = came_from.get(current)
    path.reverse()
    return path

def a_star_search(start, goal, graph):
    counter = 0
    open_set = []
    heapq.heappush(open_set, (heuristic(start, goal), counter, start))
    came_from = {start: None}
    g_cost = {start: 0}
    expansion_order = []
    closed = set()

    while open_set:
        f, _, current = heapq.heappop(open_set)
        if current in closed:
            continue
        closed.add(current)
        expansion_order.append(current)
        if current == goal:
            break
        for neighbor, cost in graph.get(current, {}).items():
            if neighbor in closed:
                continue
            new_g = g_cost[current] + cost
            if neighbor not in g_cost or new_g < g_cost[neighbor]:
                g_cost[neighbor] = new_g
                f_val = new_g + heuristic(neighbor, goal)
                counter += 1
                heapq.heappush(open_set, (f_val, counter, neighbor))
                came_from[neighbor] = current

    path = reconstruct_path(came_from, goal)
    total_cost = g_cost.get(goal, float('inf'))
    return path, total_cost, expansion_order, g_cost

start = "Pharmacy"
goal = "Emergency_Ward"
path, total_cost, expansion_order, g_cost = a_star_search(start, goal, hospital_graph)

print("A* Search")
print("--------------------------------")
print("Expansion order:")
print(" -> ".join(expansion_order))
print("\nSolution path:")
print(" -> ".join(path))
print(f"\nTotal path cost: {total_cost:.2f}")

#### g(n), h(n), f(n) table

In [ ]:
print("\nA* Node Information")
print("*********************")
print(f"{'Node':20s}{'g(n)':>10s}{'h(n)':>10s}{'f(n)':>10s}")
for node in expansion_order:
    g = g_cost[node]
    h = heuristic(node, goal)
    f = g + h
    print(f"{node:20s}{g:10.2f}{h:10.2f}{f:10.2f}")

#### NetworkX visualization

In [ ]:
G = nx.DiGraph()
for node, neighbors in hospital_graph.items():
    for neighbor, weight in neighbors.items():
        G.add_edge(node, neighbor, weight=weight)

pos = locations
path_edges = list(zip(path, path[1:]))
node_colors = ["orange" if n in path else "lightblue" for n in G.nodes()]
edge_colors = ["red" if e in path_edges else "gray" for e in G.edges()]

plt.figure(figsize=(12, 7))
nx.draw(G, pos, with_labels=True, node_color=node_colors, edge_color=edge_colors,
        node_size=2000, font_size=8, arrows=True)
edge_labels = nx.get_edge_attributes(G, "weight")
nx.draw_networkx_edge_labels(G, pos, edge_labels)
plt.title("Emergency Supply Robot - A* Search")
plt.axis("off")
plt.show()

GBFS would primarily ask:

"Which node appears closest to the Emergency Ward?"

A* instead asks:

"Considering both the cost already travelled and the estimated remaining cost, which node has the lowest estimated total cost?"

### Task 4: Weighted A* for Autonomous Delivery Drone

In [ ]:
import math
import heapq
import pandas as pd

locations = {
    "Distribution_Center": (0, 0),
    "Zone_A": (2, 1),
    "Zone_B": (1, 4),
    "Zone_C": (1, 6),
    "Zone_D": (3, 6),
    "Customer_Building": (7, 6)
}

graph = {
    "Distribution_Center": {"Zone_A": 2.2, "Zone_B": 3.0},
    "Zone_A": {"Zone_C": 2.2},
    "Zone_B": {"Zone_D": 5.0},
    "Zone_C": {"Customer_Building": 6.0},
    "Zone_D": {"Customer_Building": 3.2},
    "Customer_Building": {}
}

def heuristic(node, goal):
    x1, y1 = locations[node]
    x2, y2 = locations[goal]
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

goal = "Customer_Building"

heuristic_data = [{"Node": n, "h(n)": round(heuristic(n, goal), 2)} for n in locations]
heuristic_df = pd.DataFrame(heuristic_data)
print("Heuristic Values:")
print(heuristic_df.to_string(index=False))

In [ ]:
def weighted_a_star(start, goal, graph, weight):
    counter = 0
    open_set = []
    heapq.heappush(open_set, (0, counter, start))
    came_from = {start: None}
    g_cost = {start: 0}
    expansion_order = []
    closed = set()

    while open_set:
        f, _, current = heapq.heappop(open_set)
        if current in closed:
            continue
        closed.add(current)
        expansion_order.append(current)
        if current == goal:
            break
        for neighbor, cost in graph.get(current, {}).items():
            if neighbor in closed:
                continue
            new_g = g_cost[current] + cost
            if neighbor not in g_cost or new_g < g_cost[neighbor]:
                g_cost[neighbor] = new_g
                f_val = new_g + weight * heuristic(neighbor, goal)
                counter += 1
                heapq.heappush(open_set, (f_val, counter, neighbor))
                came_from[neighbor] = current

    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = came_from.get(current)
    path.reverse()

    return path, g_cost.get(goal, float('inf')), expansion_order

start = "Distribution_Center"
goal = "Customer_Building"
weights = [1, 1.5, 2, 3]
results = []

for w in weights:
    path, cost, expansion_order = weighted_a_star(start, goal, graph, w)
    results.append({
        "Weight": w,
        "Solution Path": " \u2192 ".join(path),
        "Total Cost": round(cost, 2),
        "Nodes Expanded": len(expansion_order),
        "Expansion Order": " \u2192 ".join(expansion_order)
    })

results_df = pd.DataFrame(results)
print(results_df[["Weight", "Solution Path", "Total Cost", "Nodes Expanded"]].to_string(index=False))

In [ ]:
print("\nDetailed Expansion Orders:")
for r in results:
    print(f"\nw = {r['Weight']}: {r['Expansion Order']}")

#### NetworkX visualization

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
for node, position in locations.items():
    G.add_node(node, pos=position)
for node, neighbors in graph.items():
    for neighbor, cost in neighbors.items():
        G.add_edge(node, neighbor, weight=cost)

pos = locations

for w in weights:
    path, cost, _ = weighted_a_star(start, goal, graph, w)
    path_edges = list(zip(path, path[1:]))
    node_colors = ["orange" if n in path else "lightblue" for n in G.nodes()]
    edge_colors = ["red" if e in path_edges else "gray" for e in G.edges()]

    plt.figure(figsize=(12, 7))
    nx.draw(G, pos, with_labels=True, node_color=node_colors, edge_color=edge_colors,
            node_size=2000, font_size=8, arrows=True)
    edge_labels = nx.get_edge_attributes(G, "weight")
    nx.draw_networkx_edge_labels(G, pos, edge_labels)
    plt.title(f"Weighted A* Solution Path | w = {w} | Total Cost = {cost:.2f}")
    plt.axis("off")
    plt.show()

#### Expansion order

This means:

In what order did the search algorithm remove/select nodes from the frontier and process them?

For example:

Expanded:
S → A → B → C → D → G

The algorithm may explore some nodes that are ultimately not part of the final route.

#### Solution path

This means:

Which nodes form the final route from start to goal?

For example:

Solution:
S → A → C → G

Here:

Expansion order:  S → A → B → C → D → G

Solution path:    S → A → C → G